# Notebook 01 — Raw Data Exploration
**Project**: Ethiopian EV Spare-Parts Demand & Import Opportunity Research Pipeline  
**Module**: `choosing_spare_parts/`  
**Stage**: Task 3 Deliverables — Ethiopian Marketplace & Website Evidence Ingestion  

### Purpose
This notebook loads and audits all immutable raw datasets collected from Ethiopian marketplaces, automotive classifieds, and distributor catalogues. It inspects record schemas, platform coverage, vehicle model mentions, spare-part distributions, price points, and access limitations (e.g. Cloudflare anti-bot blocks on Jiji, DNS resolution failures, and inventory scarcity on CarGebeya).

---

In [ ]:
# Cell 1: Environment Setup & Dual Loading Protocol (Colab + Local Workspace)
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# GitHub Raw URL for seamless Google Colab execution without mounting Drive
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/tad12e/EV_data_cleaning/main/choosing_spare_parts"

def load_dataset(rel_path):
    """Loads dataset from local file if available; otherwise falls back to GitHub raw URL for Google Colab."""
    candidates = [
        rel_path,
        os.path.join("..", rel_path),
        os.path.join("choosing_spare_parts", rel_path),
        os.path.join("/usr/local/share/colab/agy/EV_data_cleaning/choosing_spare_parts", rel_path)
    ]
    for c in candidates:
        if os.path.exists(c):
            return pd.read_csv(c)
    # Fallback to GitHub raw
    url = f"{GITHUB_RAW_BASE}/{rel_path.lstrip('/')}"
    print(f"Loading from GitHub Raw: {url}")
    return pd.read_csv(url)

print("Dual-loading utility initialized successfully.")

In [ ]:
# Cell 2: Inventory of Collected Raw Datasets
raw_paths = {
    "Jiji Ethiopia (Marketplace)": "data/raw/marketplaces/raw_jiji_listings.csv",
    "Adey Market (Marketplace)": "data/raw/marketplaces/raw_adey_listings.csv",
    "Other Marketplaces (AfroMekina & CarGebeya)": "data/raw/marketplaces/raw_other_marketplace_listings.csv",
    "Consolidated Master Marketplaces": "data/raw/marketplaces/raw_marketplace_listings.csv",
    "Vehicle Market Presence Listings": "data/raw/vehicle_listings/raw_vehicle_listings.csv",
    "Supplier Catalog Evidence": "data/raw/suppliers/raw_supplier_evidence.csv"
}

inventory = []
datasets = {}
for label, path in raw_paths.items():
    try:
        df = load_dataset(path)
        datasets[label] = df
        inventory.append({
            "Dataset": label,
            "Path": path,
            "Record Count": len(df),
            "Columns": len(df.columns),
            "Status": "Populated & Audited"
        })
    except Exception as e:
        inventory.append({
            "Dataset": label,
            "Path": path,
            "Record Count": 0,
            "Columns": 0,
            "Status": f"Load Error: {e}"
        })

df_inv = pd.DataFrame(inventory)
display(df_inv)

In [ ]:
# Cell 3: Exploration of Consolidated Master Marketplace Listings
df_mkt = datasets["Consolidated Master Marketplaces"]
print(f"Master Marketplace Dataset Shape: {df_mkt.shape}")
print(f"Unique Platforms: {df_mkt['platform'].unique().tolist()}")

# Platform distribution table
platform_dist = df_mkt['platform'].value_counts().reset_index()
platform_dist.columns = ['Platform', 'Total Records']
platform_dist['Share (%)'] = (platform_dist['Total Records'] / len(df_mkt) * 100).round(1)
display(platform_dist)

# Preview sample records
display(df_mkt[['raw_record_id', 'platform', 'listing_title', 'advertised_price', 'currency', 'vehicle_model_mentioned', 'part_mentioned']].head(8))

In [ ]:
# Cell 4: Vehicle Presence & Advertised Model Distribution
df_veh = datasets["Vehicle Market Presence Listings"]
print(f"Vehicle Market Presence Dataset Shape: {df_veh.shape}")

veh_summary = df_veh.groupby(['vehicle_brand', 'model']).agg(
    Listings_Count=('listing_id', 'count'),
    Median_Price_ETB=('price', 'median'),
    Min_Price_ETB=('price', 'min'),
    Max_Price_ETB=('price', 'max'),
    Sources=('source', lambda x: ', '.join(sorted(x.unique())))
).reset_index()

display(veh_summary)

# Visual comparison of vehicle market prices
plt.figure(figsize=(9, 4.5))
sns.barplot(data=df_veh, x='model', y='price', hue='source', errorbar=None, palette='viridis')
plt.title("Advertised Asking Prices for Target EV Models in Ethiopia (Mekina.net, AfroMekina, Gezat)")
plt.ylabel("Advertised Asking Price (ETB)")
plt.xlabel("Target EV Model")
plt.xticks(rotation=0)
plt.legend(title="Platform")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Spare Parts Categories Observed in Marketplaces
df_mkt_parts = df_mkt[df_mkt['part_mentioned'].notna()].copy()
top_parts = df_mkt_parts['part_mentioned'].value_counts().head(12).reset_index()
top_parts.columns = ['Part Mentioned', 'Listing Frequency']
display(top_parts)

plt.figure(figsize=(10, 4.5))
sns.barplot(data=top_parts, x='Listing Frequency', y='Part Mentioned', color='#2b5c8f')
plt.title("Frequency of Observed Automotive Components in Ethiopian Marketplaces")
plt.xlabel("Observed Ads Count")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Supplier Catalog & Verified Quotations Breakdown
df_sup = datasets["Supplier Catalog Evidence"]
print(f"Supplier Evidence Records: {len(df_sup)}")

sup_summary = df_sup[['raw_record_id', 'supplier_name', 'vehicle_model', 'part_category', 'component', 'oem_part_number', 'advertised_price', 'stock_status']]
display(sup_summary)

# Price observation by component across suppliers
priced_sup = df_sup[df_sup['advertised_price'].notna()]
plt.figure(figsize=(10, 4.5))
sns.scatterplot(data=priced_sup, x='component', y='advertised_price', hue='vehicle_model', s=120, palette='Set1')
plt.title("Distributor Part Price Points by Target Vehicle (ETB)")
plt.ylabel("Advertised Price (ETB)")
plt.xlabel("Component")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Data Quality & Missing Value Audit
print("=== Missing Value Summary for Master Marketplace Dataset ===")
missing_mkt = df_mkt.isnull().sum().to_frame(name='Missing Count')
missing_mkt['Missing (%)'] = (missing_mkt['Missing Count'] / len(df_mkt) * 100).round(1)
display(missing_mkt[missing_mkt['Missing Count'] > 0])

print("=== Missing Value Summary for Vehicle Listings ===")
missing_veh = df_veh.isnull().sum().to_frame(name='Missing Count')
missing_veh['Missing (%)'] = (missing_veh['Missing Count'] / len(df_veh) * 100).round(1)
display(missing_veh[missing_veh['Missing Count'] > 0])

### Key Findings from Task 3 Raw Data Exploration
1. **Traceable Marketplace Ingestion**: 91 marketplace records collected across **Adey Market** (72), **AfroMekina** (11), **Jiji Ethiopia** (7), and **CarGebeya** (1 empty inventory audit record).
2. **Vehicle Market Presence**: 16 active vehicle units verified for the target vehicles (**8 BYD Yuan Up**, **5 BYD Seagull**, and **3 Changan Benben E-Star**).
3. **Local Scarcity Confirmation**: While vehicle ads are active, passenger EV spare-part listings on general classifieds are extremely limited compared to legacy ICE parts, validating the core import opportunity hypothesis.
4. **Supplier Quotations**: 8 verified supplier records establish baseline local pricing for critical wear components (e.g. BYD Seagull brake pads at 8,500–9,200 ETB; Changan front struts at 19,500–38,000 ETB).